# 16 — Generate both input arms, with resumable per-record outputs

Use a Biowulf GPU allocation compatible with Notebook 05. Start a fresh kernel.
Both arms are regenerated; old image-arm outputs are NOT substituted because the prompt is new.
A failed record is never marked complete. Restart this notebook to skip verified saved records.
Do not run two workers for the same model and RUN_NAME concurrently.

In [ ]:
from pathlib import Path
import os, sys, json

candidate = Path(os.environ.get("JAMIA_RERUN_DIR", Path.cwd())).resolve()
if not (candidate / "rerun_config.json").is_file() and (candidate.parent / "rerun_config.json").is_file():
    candidate = candidate.parent
RERUN_DIR = candidate
if not (RERUN_DIR / "rerun_config.json").is_file():
    raise FileNotFoundError("Copy rerun_config.example.json to rerun_config.json, configure paths, and set JAMIA_RERUN_DIR to the repository root")
implementation = (RERUN_DIR / "src" / "rerun_code").resolve()
sys.path[:] = [entry for entry in sys.path if Path(entry or ".").resolve() != implementation]
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config
_, CONFIG = load_config(RERUN_DIR)
RESULTS_ROOT = Path(os.environ.get("JAMIA_OUTPUT_ROOT", CONFIG["output_root"]))
CONFIG["output_root"] = str(RESULTS_ROOT)
RUN_NAME = os.environ.get("JAMIA_ABLATION_RUN", "matched_input_v1")
ABLATION_ROOT = RESULTS_ROOT / "post_rerun" / "modality_ablation" / RUN_NAME
print("Code:", RERUN_DIR)
print("Existing inputs:", RESULTS_ROOT)
print("NEW output:", ABLATION_ROOT)


In [ ]:
# Optional Hugging Face authentication. Never paste or save a token in this notebook.
LOGIN_TO_HUGGINGFACE = False
if LOGIN_TO_HUGGINGFACE:
    from huggingface_hub import login
    from getpass import getpass
    token = os.environ.get("HF_TOKEN") or getpass("Hugging Face token (hidden): ")
    login(token=token, add_to_git_credential=False)
    del token

from rerun_code.modality_ablation import (
    load_experiment, run_generation, activate_empty_recovery_amendment,
)
# Explicitly records the approved recovery amendment for an existing v1 run.
# Does NOT change plan.json or rewrite any completed MedGemma/Phi records.
# Stop other workers on this RUN_NAME before this one-time migration.
APPLY_EMPTY_RECOVERY_AMENDMENT = True
if APPLY_EMPTY_RECOVERY_AMENDMENT:
    activate_empty_recovery_amendment(RERUN_DIR, CONFIG, RUN_NAME)
output, PLAN, _ = load_experiment(RERUN_DIR, CONFIG, RUN_NAME)
print("Frozen final-report count:", PLAN["expected_records"])
# Execution subsets only; they do not reduce the completeness requirement in Notebook 17.
SELECTED_MODELS = PLAN["models"]  # e.g. ["medgemma_4b"]
SELECTED_BUNDLES = PLAN["bundles"]
RUN_GENERATION = False  # set True after inspecting the frozen plan and obtaining a GPU

In [ ]:
if not RUN_GENERATION:
    print("Not started. Set RUN_GENERATION=True in the preceding cell, then run both cells.")
else:
    run_generation(RERUN_DIR, CONFIG, RUN_NAME,
                   selected_models=SELECTED_MODELS, selected_bundles=SELECTED_BUNDLES)
    print("Requested execution subset finished. Run other frozen models/bundles before Notebook 17.")